In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

base_dir = Path("../input")
if not (base_dir / "ventilator-pressure-prediction").exists():
    base_dir = Path("./input")
train_path = base_dir / "ventilator-pressure-prediction" / "train.csv"
test_path = base_dir / "ventilator-pressure-prediction" / "test.csv"

dtypes = {
    "R": np.int8,
    "C": np.int8,
    "breath_id": np.int32,
    "id": np.int16,
    "time_step": np.float32,
    "u_in": np.float32,
    "u_out": np.int8,
    "pressure": np.float32,
}
train_df = pd.read_csv(train_path, dtype=dtypes)
test_df = pd.read_csv(test_path, dtype=dtypes)




In [2]:
def add_features(df):
    grp = df.groupby("breath_id", sort=False)

    df["cum_u_in"] = grp["u_in"].cumsum()
    df["cum_u_out"] = grp["u_out"].cumsum()
    df["delta_u_in"] = grp["u_in"].diff().fillna(0)
    df["delta_u_out"] = grp["u_out"].diff().fillna(0)
    df["prev_u_in"] = grp["u_in"].shift(1).fillna(0)
    df["prev_u_out"] = grp["u_out"].shift(1).fillna(0)
    df["time_step_diff"] = grp["time_step"].diff().fillna(0)

    stats = grp.agg(
        breath_len=("time_step", "count"),
        mean_u_in=("u_in", "mean"),
        max_u_in=("u_in", "max"),
        min_u_in=("u_in", "min"),
        mean_u_out=("u_out", "mean"),
        max_u_out=("u_out", "max"),
        min_u_out=("u_out", "min"),
    ).reset_index()

    df = df.merge(stats, on="breath_id", how="left")

    df["R_mul_C"] = df["R"] * df["C"]
    df["u_in_mul_u_out"] = df["u_in"] * df["u_out"]
    df["time_mul_u_in"] = df["time_step"] * df["u_in"]
    df["time_bin"] = (df["time_step"] * 100).round().astype(np.int16)

    # ---- additional cheap features ----
    df["R_div_C"] = df["R"] / df["C"]
    df["u_in_sq"] = df["u_in"] ** 2
    df["time_step_sq"] = df["time_step"] ** 2
    # normalized cumulative u_in (per breath)
    df["cum_u_in_ratio"] = df["cum_u_in"] / df["breath_len"]
    # end of additional features ----

    return df


train_df = add_features(train_df)
test_df = add_features(test_df)

feature_cols = [
    "R",
    "C",
    "u_in",
    "u_out",
    "time_step",
    "cum_u_in",
    "cum_u_out",
    "delta_u_in",
    "delta_u_out",
    "prev_u_in",
    "prev_u_out",
    "time_step_diff",
    "breath_len",
    "mean_u_in",
    "max_u_in",
    "min_u_in",
    "mean_u_out",
    "max_u_out",
    "min_u_out",
    "R_mul_C",
    "u_in_mul_u_out",
    "time_mul_u_in",
    "time_bin",
    # newly added features
    "R_div_C",
    "u_in_sq",
    "time_step_sq",
    "cum_u_in_ratio",
]

X = train_df[feature_cols].values.astype(np.float32)
y = train_df["pressure"].values.astype(np.float32)



In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
from sklearn.ensemble import HistGradientBoostingRegressor


def train_hgb(X_train, y_train, X_val, y_val):
    hgb = HistGradientBoostingRegressor(
        loss="least_absolute_deviation",
        max_iter=600,
        learning_rate=0.04,
        max_depth=12,
        max_bins=255,
        min_samples_leaf=10,
        random_state=42,
        early_stopping=True,
    )
    hgb.fit(X_train, y_train)
    val_pred = hgb.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    print(f"Validation MAE (HistGradientBoostingRegressor): {val_mae:.5f}")
    return hgb


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

try:
    model = train_hgb(X_train, y_train, X_val, y_val)
    # Re‑fit on the full data for final predictions
    model.fit(X, y)

    test_features = test_df[feature_cols].values.astype(np.float32)
    model_pred = model.predict(test_features)

    # simple R‑C‑time_bin baseline
    pressure_lookup = (
        train_df[["R", "C", "time_bin", "pressure"]]
        .groupby(["R", "C", "time_bin"], as_index=False)
        .mean()
    )
    baseline_pred = (
        test_df[["R", "C", "time_bin"]]
        .merge(pressure_lookup, on=["R", "C", "time_bin"], how="left")["pressure"]
        .fillna(train_df["pressure"].mean())
        .values
    )

    # rely more on the model (80 % model, 20 % baseline)
    test_pred = 0.8 * model_pred + 0.2 * baseline_pred

except Exception as e:
    print("Model training failed, falling back to simple R‑C mean baseline.")
    print(f"Error: {e}")

    rc_mean = train_df.groupby(["R", "C"])["pressure"].mean().reset_index()
    test_pred = (
        test_df[["R", "C"]]
        .merge(rc_mean, on=["R", "C"], how="left")["pressure"]
        .fillna(train_df["pressure"].mean())
        .values
    )

test_pred = np.clip(test_pred, a_min=0, a_max=None)



/usr/local/lib/python3.11/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:16: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


Model training failed, falling back to simple R‑C mean baseline.
Error: The 'loss' parameter of HistGradientBoostingRegressor must be a str among {'poisson', 'absolute_error', 'squared_error', 'quantile'} or an instance of 'sklearn._loss.loss.BaseLoss'. Got 'least_absolute_deviation' instead.


In [4]:
submission = test_df[["id"]].copy()
submission["pressure"] = test_pred.astype(float)
submission = submission[["id", "pressure"]]



In [5]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")
print(submission.head())

Submission written to submission.csv
   id   pressure
0   1  12.102113
1   2  12.102113
2   3  12.102113
3   4  12.102113
4   5  12.102113
